# 유틸리티

In [4]:
!pip install ultralytics

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


In [11]:
import torch
from ultralytics import YOLO

In [3]:
# GPU 사용확인

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    print('Using PyTorcha version:', torch.__version__, ' Device:', DEVICE)
    print('CUDA is available. Using GPU for training.')
else:
    DEVICE = torch.device('cpu')
    print('Using PyTorch version:', torch.__version__, ' Device:', DEVICE)
    print('CUDA is not available. Using CPU for training.')

Using PyTorcha version: 2.8.0+cu128  Device: cuda
CUDA is available. Using GPU for training.


# YOLO train을 위한 Json -> txt 및 Yaml 파일 생성

In [ ]:
# json->txt 변환 (이후로 실행할 필요 없음)

import os, json

# === 경로 정확히 반영 ===
SPLITS = [
    dict(
        img="/home/work/root/Meme_coin/Data/Training/img/TS_KS",
        lbl="/home/work/root/Meme_coin/Data/Training/label/TL_KS_BBOX",  # ← train 라벨
        out="/home/work/root/Meme_coin/Data/YOLO_labels/Training/TS_KS"
    ),
    dict(
        img="/home/work/root/Meme_coin/Data/Validation/img/VS_KS",
        lbl="/home/work/root/Meme_coin/Data/Validation/label/VL_KS_BBOX",  # ← val 라벨
        out="/home/work/root/Meme_coin/Data/YOLO_labels/Validation/VS_KS"
    ),
]

CLASSES = ["Chimney"]  # 단일 클래스
name2id = {n: i for i, n in enumerate(CLASSES)}

def convert_split(img_dir, lbl_dir, out_dir):
    if not os.path.isdir(lbl_dir):
        raise FileNotFoundError(f"라벨 폴더 없음: {lbl_dir}")
    os.makedirs(out_dir, exist_ok=True)

    made, skipped = 0, 0
    for jf in sorted(os.listdir(lbl_dir)):
        if not jf.endswith(".json"):
            continue
        with open(os.path.join(lbl_dir, jf), "r", encoding="utf-8") as f:
            data = json.load(f)

        for _, item in data.items():  # VIA: top-level key마다 1 이미지
            fname = item.get("filename")
            if not fname:
                skipped += 1
                continue

            # 이미지 크기 (VIA file_attributes에서 제공)
            fa = item.get("file_attributes", {})
            try:
                w = int(fa.get("img_width"))   # "512" → 512
                h = int(fa.get("img_height"))
            except Exception:
                print(f"⚠ 이미지 크기 없음: {fname} (file_attributes 확인 필요)")
                skipped += 1
                continue

            # bbox들을 YOLO 형식으로 변환
            lines = []
            for reg in item.get("regions", []):
                sa = reg.get("shape_attributes", {})
                if sa.get("name") not in ("rect", "rectangle"):
                    continue
                x, y = float(sa["x"]), float(sa["y"])
                bw, bh = float(sa["width"]), float(sa["height"])
                cx = (x + bw / 2.0) / w
                cy = (y + bh / 2.0) / h
                ww = bw / w
                hh = bh / h
                lines.append(f"0 {cx:.6f} {cy:.6f} {ww:.6f} {hh:.6f}")  # Chimney=0

            # 출력
            base = os.path.splitext(os.path.basename(fname))[0]
            out_txt = os.path.join(out_dir, base + ".txt")
            with open(out_txt, "w", encoding="utf-8") as f:
                f.write("\n".join(lines))
            print("✅", out_txt, f"({len(lines)} boxes)")
            made += 1

    print(f"[완료] {lbl_dir} → {out_dir} : 만든 파일 {made}개, 스킵 {skipped}개")

if __name__ == "__main__":
    for s in SPLITS:
        print(f"\n=== split: {s['lbl']} ===")
        convert_split(s["img"], s["lbl"], s["out"])

In [6]:
# YOLO 모델 실행을 위하여 yaml 파일 생성

import yaml

data_yaml = {
    "train": "/home/work/root/Meme_coin/Data/Training/images",
    "val": "/home/work/root/Meme_coin/Data/Validation/images",
    "nc": 1,
    "names": ["Chimney"]
}

yaml_path = "/home/work/root/Meme_coin/data.yaml"
with open(yaml_path, "w") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)

# 확인
print(open(yaml_path).read())

train: /home/work/root/Meme_coin/Data/Training/images
val: /home/work/root/Meme_coin/Data/Validation/images
nc: 1
names:
- Chimney



# 최종 모델 (train 및 평가)

In [4]:
#"v8x_e200_i512_not_pct"

from ultralytics import YOLO


model = YOLO("yolov8x.pt")

# 학습 안정성 
model.train(
    data="/home/work/root/Meme_coin/data.yaml",
    epochs=200,
    imgsz=512,          # 입력 이미지 크기
    batch=32,           # 배치 크기
    workers=0,          # dataloader worker (주피터에서 안정적)
    amp = False,
    device=0,        
    exist_ok=True,      # 기존 결과 덮어쓰기 허용
    name="v8x_e200_i512_not_pct",
    seed=42   # 시드 고정
)

New https://pypi.org/project/ultralytics/8.3.206 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.198 🚀 Python-3.12.3 torch-2.8.0+cu128 CUDA:0 (NVIDIA H100 80GB HBM3, 57091MiB)
engine/trainer: agnostic_nms=False, amp=False, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/work/root/Meme_coin/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8x.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=v8x_e200_i512_not_p

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7fd656300b00>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [10]:
from ultralytics import YOLO

# 1. 학습된 모델 로드
model = YOLO("/home/work/root/Meme_coin/runs/detect/v8x_e200_i512_not_pct/weights/best.pt")

# 2. Validation 평가 실행 (Mission 1 설정 반영)
results = model.val(
    data="/home/work/root/Meme_coin/data.yaml",  # validation 세트 정의 파일
    split="val",        # 검증 세트 사용
    imgsz=512,          # 평가 시 입력 이미지 크기 (학습 시와 동일)
    batch=32,           # GPU VRAM 여유에 따라 조정
    device=0,           # GPU 번호
    workers=0,          # dataloader worker 수
    save_json=True,     # COCO metrics용 json 저장
    save_hybrid=False,  # hybrid predictions 저장 여부
    verbose=True,       # 로그 상세 출력

    # ===== Mission 1 추가 설정 =====
    conf=0.25,          # confidence threshold
    iou=0.50,           # NMS IoU threshold
    max_det=100         # 최대 탐지 수
)


WARNING ⚠️ 'save_hybrid' is deprecated and will be removed in the future.
Ultralytics 8.3.198 🚀 Python-3.12.3 torch-2.8.0+cu128 CUDA:0 (NVIDIA H100 80GB HBM3, 40780MiB)
Model summary (fused): 112 layers, 68,124,531 parameters, 0 gradients, 257.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 637.0±90.7 MB/s, size: 109.1 KB)
val: Scanning /home/work/root/Meme_coin/Data/Validation/labels/VS_KS.cache... 1006 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1006/1006 2.9Mit/s 0.0s0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 32/32 3.9it/s 8.2s0.2s
                   all       1006       1322      0.986      0.987      0.993      0.905
Speed: 0.0ms preprocess, 2.5ms inference, 0.0ms loss, 0.4ms postprocess per image
Saving /home/work/root/Meme_coin/runs/detect/val49/predictions.json...
Results saved to /home/work/root/Meme_coin/runs/detect/val49


# 기타 여러가지 모델 실험 시행착오 (최종 모델 X)

In [5]:
#"v8x_e100_i512_not_pct_2"

from ultralytics import YOLO


model = YOLO("yolov8x.pt")

# 학습 안정성 + Early Stopping
model.train(
    data="/home/work/root/Meme_coin/data.yaml",
    epochs=100,
    imgsz=512,          # 입력 이미지 크기
    batch=64,           # 배치 크기
    workers=0,          # dataloader worker (주피터에서 안정적)
    amp = False,
    device=0,        
    exist_ok=True,      # 기존 결과 덮어쓰기 허용
    name="v8x_e100_i512_not_pct_2",
    seed=42   # 🔑 시드 고정
)



New https://pypi.org/project/ultralytics/8.3.202 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.198 🚀 Python-3.12.3 torch-2.8.0+cu128 CUDA:0 (NVIDIA H100 80GB HBM3, 81090MiB)
engine/trainer: agnostic_nms=False, amp=False, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/work/root/Meme_coin/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8x.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=v8x_e100_i512_not_p

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7f4b5c18c3b0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [14]:
# "v8n_e100_i512_not_pct" (0.969/0.745)

from ultralytics import YOLO


model = YOLO("yolov8n.pt")

# 학습 안정성 + Early Stopping
model.train(
    data="/home/work/root/Meme_coin/data.yaml",
    epochs=100,
    imgsz=512,          # 입력 이미지 크기
    batch=128,           # 배치 크기
    workers=0,          # dataloader worker (주피터에서 안정적)
    device=0,        
    exist_ok=True,      # 기존 결과 덮어쓰기 허용
    name="v8n_e100_i512_not_pct",
    seed=42,   # 🔑 시드 고정
    patience=5         # Early Stopping (5번 연속 개선 없으면 멈춤)
)

New https://pypi.org/project/ultralytics/8.3.199 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.198 🚀 Python-3.12.3 torch-2.6.0+cu124 CUDA:0 (NVIDIA H100 80GB HBM3, 57091MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=128, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/work/root/Meme_coin/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=v8n_e100_i512_not_p

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7f16246d7470>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [5]:
#"v8s_e100_i640_not_pct" 

from ultralytics import YOLO


model = YOLO("yolov8s.pt")

# 학습 안정성 + Early Stopping
model.train(
    data="/home/work/root/Meme_coin/data.yaml",
    epochs=100,
    imgsz=640,          # 입력 이미지 크기
    batch=64,           # 배치 크기
    workers=0,          # dataloader worker (주피터에서 안정적)
    device=0,        
    exist_ok=True,      # 기존 결과 덮어쓰기 허용
    name="v8s_e100_i640_not_pct",
    seed=42,   # 🔑 시드 고정
    patience=5         # Early Stopping (5번 연속 개선 없으면 멈춤)
)

New https://pypi.org/project/ultralytics/8.3.199 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.198 🚀 Python-3.12.3 torch-2.6.0+cu124 CUDA:0 (NVIDIA H100 80GB HBM3, 16312MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/work/root/Meme_coin/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=v8s_e100_i640_not_pc

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7f4867aa6d20>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [7]:
#"v8m_e100_i640_not_pct" 

from ultralytics import YOLO


model = YOLO("yolov8m.pt")

# 학습 안정성 + Early Stopping
model.train(
    data="/home/work/root/Meme_coin/data.yaml",
    epochs=100,
    imgsz=640,          # 입력 이미지 크기
    batch=16,           # 배치 크기
    workers=0,          # dataloader worker (주피터에서 안정적)
    device=0,        
    exist_ok=True,      # 기존 결과 덮어쓰기 허용
    name="v8m_e100_i640_not_pct",
    seed=42,   # 🔑 시드 고정
    patience=5         # Early Stopping (5번 연속 개선 없으면 멈춤)
)

New https://pypi.org/project/ultralytics/8.3.199 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.198 🚀 Python-3.12.3 torch-2.6.0+cu124 CUDA:0 (NVIDIA H100 80GB HBM3, 8156MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/work/root/Meme_coin/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=v8m_e100_i640_not_pct

train: Scanning /home/work/root/Meme_coin/Data/Training/labels/TS_KS.cache... 8052 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 8052/8052 7.2Mit/s 0.0s
val: Fast image access ✅ (ping: 0.1±0.1 ms, read: 572.0±87.3 MB/s, size: 114.6 KB)
val: Scanning /home/work/root/Meme_coin/Data/Validation/labels/VS_KS.cache... 1006 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1006/1006 1.9Mit/s 0.0s0s
Plotting labels to /home/work/root/Meme_coin/runs/detect/v8m_e100_i640_not_pct/labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: SGD(lr=0.01, momentum=0.9) with parameter groups 77 weight(decay=0.0), 84 weight(decay=0.0005), 83 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 0 dataloader workers
Logging results to /home/work/root/Meme_coin/runs/detect/v8m_e100_i640_not_pct
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_los

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7f0ca4cdc650>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [7]:
#"v8x_e100_i512_not_pct"

from ultralytics import YOLO


model = YOLO("yolov8x.pt")


model.train(
    data="/home/work/root/Meme_coin/data.yaml",
    epochs=100,
    imgsz=512,          # 입력 이미지 크기
    batch=32,           # 배치 크기
    workers=0,          # dataloader worker (주피터에서 안정적)
    device=0,        
    exist_ok=True,      # 기존 결과 덮어쓰기 허용
    name="v8x_e100_i512_not_pct",
    seed=42,   # 🔑 시드 고정
    patience=5         # Early Stopping (5번 연속 개선 없으면 멈춤)
)

New https://pypi.org/project/ultralytics/8.3.199 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.198 🚀 Python-3.12.3 torch-2.6.0+cu124 CUDA:0 (NVIDIA H100 80GB HBM3, 16312MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/work/root/Meme_coin/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8x.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=v8x_e100_i512_not_pc

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7fe690c72510>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 